# Phase 1 -- Exploratory Data Analysis

Dataset: **IEEE-CIS Fraud Detection** (590,540 transactions, 3.5% fraud rate).

Covers the Phase 1 EDA acceptance criteria:

1. Class distribution (fraud vs. non-fraud).
2. Missing-value heatmap across feature groups.
3. Amount distributions (raw + log-transformed).
4. Top-correlated V-features vs. `isFraud`.
5. Temporal fraud rate.
6. Device / email domain breakdowns.

> Run `make download-data` before opening this notebook -- it reads the CSVs from `data/raw/`.

## 0. Setup

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

# Make the src/ package importable without installing.
ROOT = Path.cwd().resolve()
if (ROOT / "src").exists():
    src = ROOT / "src"
elif (ROOT.parent / "src").exists():
    src = ROOT.parent / "src"
else:
    raise RuntimeError("Could not locate src/ relative to this notebook.")
sys.path.insert(0, str(src))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from fraud_detection.utils.config import load_config
from fraud_detection.data.preprocessing import group_columns, AMOUNT_COL

sns.set_theme(style="whitegrid", context="talk")
plt.rcParams["figure.dpi"] = 110

cfg = load_config()
RAW_DIR = cfg.paths.data_raw
TARGET = cfg.dataset.target
TIME = cfg.dataset.time_column
print(f"Project root: {cfg.paths.data_raw.parents[1]}")
print(f"Raw data dir: {RAW_DIR}")

## 1. Load and join transaction + identity tables

In [ ]:
tx_path = RAW_DIR / cfg.dataset.transaction_file
id_path = RAW_DIR / cfg.dataset.identity_file

if not tx_path.exists():
    raise FileNotFoundError(
        f"{tx_path} not found -- run `make download-data` first."
    )

# For CPU-friendliness in EDA we optionally subsample. Set USE_SUBSET=False
# to load the full 590K rows.
USE_SUBSET = cfg.dataset.use_subset
SUBSET_N = cfg.dataset.subset_size if USE_SUBSET else None

tx = pd.read_csv(tx_path, nrows=SUBSET_N)
print(f"Transactions: {tx.shape}")

if id_path.exists():
    ident = pd.read_csv(id_path, nrows=SUBSET_N)
    print(f"Identity:     {ident.shape}")
    df = tx.merge(ident, how="left", on=cfg.dataset.join_key)
else:
    df = tx
print(f"Joined:       {df.shape}")
df.head(3)

## 2. Class distribution

In [ ]:
target_counts = df[TARGET].value_counts().sort_index()
fraud_rate = df[TARGET].mean()
print(f"Fraud rate: {fraud_rate:.4%} ({target_counts.get(1, 0):,} / {len(df):,})")

fig, ax = plt.subplots(figsize=(6, 4))
sns.barplot(
    x=["Non-fraud", "Fraud"],
    y=target_counts.values,
    palette=["#4c72b0", "#c44e52"],
    ax=ax,
)
for i, v in enumerate(target_counts.values):
    ax.text(i, v, f"{v:,}", ha="center", va="bottom", fontsize=11)
ax.set_title("Class distribution")
ax.set_ylabel("count")
plt.tight_layout()

## 3. Missing values by feature group

In [ ]:
groups = group_columns(df, target=TARGET, time_col=TIME)

def _miss(cols):
    cols = [c for c in cols if c in df.columns]
    if not cols:
        return 0.0
    return df[cols].isna().mean().mean()

summary = pd.DataFrame(
    {
        "group": ["V1-V339", "D1-D15", "C1-C14", "M1-M9", "id_numeric", "id_categorical", "email"],
        "n_cols": [
            len(groups.v_features),
            len(groups.d_features),
            len(groups.c_features),
            len(groups.m_features),
            len(groups.id_numeric),
            len(groups.id_categorical),
            len(groups.email_cols),
        ],
        "avg_missing_pct": [
            _miss(groups.v_features),
            _miss(groups.d_features),
            _miss(groups.c_features),
            _miss(groups.m_features),
            _miss(groups.id_numeric),
            _miss(groups.id_categorical),
            _miss(groups.email_cols),
        ],
    }
)
summary

In [ ]:
# Heatmap of missing fraction per column, sorted by missingness.
miss_frac = df.isna().mean().sort_values(ascending=False)
top = miss_frac.head(60)
fig, ax = plt.subplots(figsize=(10, 10))
sns.heatmap(
    top.to_frame("missing_frac"),
    annot=True,
    fmt=".2f",
    cmap="Reds",
    cbar=True,
    yticklabels=True,
    xticklabels=True,
    ax=ax,
)
ax.set_title("Top 60 columns by missing fraction")
plt.tight_layout()

## 4. Amount distribution (raw + log)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.histplot(
    data=df,
    x=AMOUNT_COL,
    hue=TARGET,
    bins=80,
    log_scale=(False, True),
    palette=["#4c72b0", "#c44e52"],
    ax=axes[0],
)
axes[0].set_title("TransactionAmt (raw, log-y)")

sns.histplot(
    data=df.assign(log_amt=np.log1p(df[AMOUNT_COL].clip(lower=0))),
    x="log_amt",
    hue=TARGET,
    bins=80,
    palette=["#4c72b0", "#c44e52"],
    ax=axes[1],
)
axes[1].set_title("log1p(TransactionAmt)")
plt.tight_layout()

summary = (
    df.groupby(TARGET)[AMOUNT_COL]
    .agg(["count", "mean", "median", "std", "max"])
    .rename(index={0: "non-fraud", 1: "fraud"})
)
summary

## 5. V-feature correlations with `isFraud`

In [ ]:
v_cols = groups.v_features
if v_cols:
    # Compute absolute Pearson correlation with the target.
    with pd.option_context("mode.use_inf_as_na", True):
        corrs = df[v_cols].corrwith(df[TARGET]).abs().sort_values(ascending=False)
    top_corr = corrs.head(20)

    fig, ax = plt.subplots(figsize=(8, 6))
    sns.barplot(x=top_corr.values, y=top_corr.index, ax=ax, palette="rocket")
    ax.set_title("Top 20 |corr| of V-features with isFraud")
    ax.set_xlabel("|Pearson r|")
    plt.tight_layout()
    display(top_corr.head(20).rename("|corr|").to_frame())
else:
    print("No V-features detected.")

## 6. Temporal fraud rate

`TransactionDT` is seconds since a reference date. We'll bucket into days and compute rolling fraud rate.

In [ ]:
seconds_per_day = 60 * 60 * 24
df["_day"] = (df[TIME] // seconds_per_day).astype(int)

daily = (
    df.groupby("_day")
    .agg(n=(TARGET, "size"), fraud=(TARGET, "mean"))
    .reset_index()
)

fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
sns.lineplot(data=daily, x="_day", y="n", ax=axes[0], color="#4c72b0")
axes[0].set_title("Transactions per day")
axes[0].set_ylabel("count")

sns.lineplot(data=daily, x="_day", y="fraud", ax=axes[1], color="#c44e52")
axes[1].axhline(fraud_rate, ls="--", color="grey", alpha=0.6, label=f"overall {fraud_rate:.2%}")
axes[1].set_title("Daily fraud rate")
axes[1].set_ylabel("fraud rate")
axes[1].legend()
axes[1].set_xlabel("day (0 = start of window)")
plt.tight_layout()
df.drop(columns="_day", inplace=True)

## 7. Device / email breakdowns

Quick look at two of the strongest entity signals flagged in the plan.

In [ ]:
def _fraud_by_category(col, top_n=10):
    if col not in df.columns:
        print(f"{col} not present")
        return None
    grouped = (
        df.groupby(col, dropna=False)[TARGET]
        .agg(["size", "mean"])
        .rename(columns={"size": "n", "mean": "fraud_rate"})
        .sort_values("n", ascending=False)
        .head(top_n)
    )
    return grouped

print("Top P_emaildomain")
display(_fraud_by_category("P_emaildomain"))
print("Top DeviceType")
display(_fraud_by_category("DeviceType"))
print("Top ProductCD")
display(_fraud_by_category("ProductCD"))

## 8. Summary table

In [ ]:
summary = {
    "rows": len(df),
    "columns": df.shape[1],
    "fraud_rate": float(fraud_rate),
    "n_v_features": len(groups.v_features),
    "n_d_features": len(groups.d_features),
    "n_c_features": len(groups.c_features),
    "n_m_features": len(groups.m_features),
    "n_id_numeric": len(groups.id_numeric),
    "n_id_categorical": len(groups.id_categorical),
    "avg_missing_v": float(df[groups.v_features].isna().mean().mean()) if groups.v_features else None,
    "amount_mean": float(df[AMOUNT_COL].mean()),
    "amount_median": float(df[AMOUNT_COL].median()),
    "amount_p99": float(df[AMOUNT_COL].quantile(0.99)),
}
pd.Series(summary, name="value").to_frame()